# 🧭 生成式AI應用開發｜第04週｜150分鐘修訂版
## Prompt Engineering：從訊息結構到可驗證的小功能
延續第三周 Responses API，以固定 few-shot 範例銜接第五周對話紀錄。
本檔另存自原實作教材，原版保留。先備：list/dict、for、函式、instructions、role/content。

> **學生版**：訊息組裝、客服分類器與擇一應用保留 TODO；先完成練習再開啟付費區塊。

## 🎯 學習目標
1. 區分工作身分與訊息role，使用instructions搭配訊息列表。
2. 用五元素、分隔符與任務拆解寫出可測試的規格。
3. 完成引導練習：用迴圈組裝user/assistant範例，並離線檢查。
4. 完成練習A：客服分類器v1/v2，用同一批案例比較有無範例。
5. 完成練習B改寫器或練習C資料問答其中一題，檢查輸出與事實。
6. 分清API失敗、非法格式與答案錯誤；v3衝突範例為選做。

## 🗺️ 150分鐘課堂流程（有效授課時間，休息另計）
| 分鐘 | 區塊 | 完成要求 |
|---|---|---|
| 0–15 | 環境與第三周複習 | 規則、資料、訊息角色 |
| 15–35 | 基準比較、五元素 | 觀察與改良prompt |
| 35–55 | instructions、分隔符、任務拆解 | 讀懂並修改示範 |
| 55–80 | 短示範＋引導練習：訊息組裝 | 離線檢查9則訊息 |
| 80–110 | 練習A：客服分類器（必做） | v1單筆→v2單筆→五題比較 |
| 110–135 | 練習B改寫／練習C問答（擇一） | 所選題輸出與檢查 |
| 135–150 | 同儕覆核、修訂紀錄 | 說明結果與修改理由 |
摘要與課程意見分類只作示範；學生實作一個客服分類器，再完成B或C。
v3、未選題與額外挑戰均為延伸，不影響課堂完成判定。

---

## 1️⃣ 環境準備

沿用第 3 週設定：API key 放在 Colab Secrets 的 `OPENAI_API_KEY`，不要直接寫進 Notebook。程式會先檢查金鑰，再建立 API client；若設定不完整，會提供明確錯誤訊息。


In [ ]:
# 安裝或更新 OpenAI Python SDK
!pip install -q --upgrade openai


In [ ]:
import os
from openai import OpenAI

# Colab 優先讀取 Secrets；本機執行時則使用作業系統環境變數。
try:
    from google.colab import userdata
    secret_key = userdata.get("OPENAI_API_KEY")
    if secret_key:
        os.environ["OPENAI_API_KEY"] = secret_key
except ImportError:
    # 非 Colab 環境時，沿用作業系統環境變數。
    pass
except Exception as exc:
    print(f"無法讀取 Colab Secret：{exc}")


# 沒有金鑰也能完成訊息列表與規格的離線練習；付費入口會明確提示。
MODEL = os.getenv("OPENAI_MODEL", "gpt-5.4-mini")
client = OpenAI() if os.getenv("OPENAI_API_KEY") else None
print("本週模型：", MODEL)
print("API 已可用" if client else "離線練習模式：尚未設定 API key")

In [ ]:
def require_text(value, name):
    """在本機先拒絕空白或錯誤型別，避免無效的付費請求。"""
    if not isinstance(value, str) or not value.strip():
        raise ValueError(f"{name} 必須是非空白字串。")


def request_text(api_input, instructions):
    """共用 API 與錯誤處理；各入口先驗證自己的輸入結構。"""
    require_text(instructions, "instructions")
    if client is None:
        raise RuntimeError("請先設定 OPENAI_API_KEY 並重新執行環境 cell。")
    try:
        response = client.responses.create(
            model=MODEL, instructions=instructions, input=api_input, store=False,
        )
    except Exception as exc:
        # API 失敗與模型答錯分開處理，不把錯誤文字當成正常答案。
        raise RuntimeError(f"OpenAI API 呼叫失敗：{exc}") from exc
    if getattr(response, "status", "completed") != "completed":
        raise RuntimeError("回應尚未完成，不採用部分文字。")
    for item in getattr(response, "output", []) or []:
        for part in getattr(item, "content", []) or []:
            if getattr(part, "type", None) == "refusal":
                raise RuntimeError("模型拒絕此請求，請檢查資料與任務。")
    result = response.output_text
    if not isinstance(result, str) or not result.strip():
        raise RuntimeError("模型沒有回傳可用文字。")
    return result


def ask_ai(user_input, instructions="你是一位清楚、可靠的繁體中文助理"):
    """沿用單次文字任務入口，摘要、改寫與資料問答使用此函式。"""
    require_text(user_input, "user_input")
    return request_text(user_input, instructions)


def ask_ai_messages(messages, instructions):
    """接收有角色的訊息列表；本週只使用 user/assistant 固定範例。"""
    if not isinstance(messages, list) or not messages:
        raise ValueError("messages 必須是非空列表。")
    for message in messages:
        if not isinstance(message, dict) or message.get("role") not in {"user", "assistant"}:
            raise ValueError("本週訊息 role 必須是 user 或 assistant。")
        require_text(message.get("content"), "content")
    if messages[-1]["role"] != "user":
        raise ValueError("最後一則必須是本次待回答的 user 訊息。")
    return request_text(messages, instructions)


def compare_prompts(user_input, prompt_variants):
    """同一筆文字搭配不同 instructions；每個版本各付費一次。"""
    if not isinstance(prompt_variants, dict) or not prompt_variants:
        raise ValueError("prompt_variants 必須是非空字典。")
    return {name: ask_ai(user_input, rules) for name, rules in prompt_variants.items()}

> **執行原則**：依每段旁邊的步驟操作，只開啟當前活動的開關，不必重跑整份Notebook。
> 定義函式不付費；明列的API示範／測試才付費。完成依賴的TODO再開啟該題。
> 各段開關預設False，成本以請求數與實際usage判讀；store=False仍會傳送資料。
> 一次分類相同不代表範例無效；加入few-shot也不保證分數提高。

---

## 2️⃣ 從模糊 prompt 建立基準結果

先保留簡單版本作為 baseline。之後每次只調整一個主要變因，才能判斷是哪項改動造成差異。


In [ ]:
course_intro = """
生成式AI應用開發是一門大三課程，學生會使用 Python、OpenAI API、
Streamlit 與 GitHub 完成 AI 應用。課程重視實作、API key 安全、成本控制，
並以期中個人專題和期末小組專題整合所學。
"""

# 用 XML-like 標籤包住資料，可降低模型把指令與資料混在一起的機率。
comparison_input = f"""請摘要以下課程資料：
<course>
{course_intro}
</course>"""
# baseline prompt 故意保持簡短，稍後用來和明確版本比較差異。
baseline_instructions = "請摘要使用者提供的課程資料。"
# 本段示範：1次API請求；不依賴後面的學生練習。
RUN_BASELINE = False
if RUN_BASELINE:
    baseline_result = ask_ai(comparison_input, baseline_instructions)
    print(baseline_result)

### ✍️ 觀察紀錄

- 摘要有幾句？是否使用繁體中文？
- 是否保留課程工具與評量方式？
- 重跑一次時，長度與格式是否相同？

模糊 prompt 不一定失敗，但它沒有先定義「什麼叫成功」。


---

## 3️⃣ Prompt 的五個實用元素

| 元素 | 要回答的問題 | 範例 |
|---|---|---|
| 角色 Role | 模型以什麼身份工作？ | 大學課程助教 |
| 任務 Task | 具體完成什麼？ | 摘要課程介紹 |
| 背景 Context | 必要資料是什麼？ | `<course>...</course>` |
| 限制 Constraints | 長度、語言、禁忌？ | 60 字內、不新增資訊 |
| 輸出 Output | 結果長什麼樣？ | 一句總結＋三個重點 |

口訣：**誰、做什麼、根據什麼、遵守什麼、輸出什麼。**


In [ ]:
# 明確版本把角色、資料限制、輸出長度與重點數都寫進 instructions。
clear_instructions = """
# Identity
你是大學課程的教學助理，擅長整理繁體中文課程資訊。

# Instructions
1. 只能根據使用者提供的課程文字摘要，不得新增資訊。
2. 先寫一句 60 字以內的總結。
3. 再列三個重點，涵蓋技術、課程特色與評量。
4. 使用繁體中文。
"""
# 本段示範：1次API請求；不依賴後面的學生練習。
RUN_CLEAR_PROMPT = False
if RUN_CLEAR_PROMPT:
    clear_result = ask_ai(comparison_input, clear_instructions)
    print(clear_result)
# 已保留兩份輸出，不另重複呼叫 compare_prompts；它可供課後新版本比較。

### ✍️ 預測、觀察、修正（兩版皆填）
執行前先圈出五元素；預測明確版本可能改善哪一項，再比較實際結果。
| 面向 | 基準實際結果 | 明確版實際結果 | 通過條件 |
|---|---|---|---|
| 格式 |  |  | 一句總結＋三點 |
| 長度 |  |  | 總結 60 字元內 |
| 必要資訊 |  |  | 技術、特色、評量 |
| 事實 |  |  | 沒有新增原文未提供的資訊 |
不預填「一定改善」；結果可能相同或退步。下一版只調整一個主要變因。
字數以 Python 字元數計，包含標點；換行可排除，但不等於 token。

---

## 4️⃣ `instructions` 與 `input`：規則和資料分開

- `instructions`：應用程式提供的固定角色、規則、流程與範例。
- `input`：每次請求會改變的使用者任務與資料。

課堂常把固定高層規則泛稱為 system prompt；本課使用 Responses API，因此用 `instructions` 實作。可把固定規則想成函式定義，把每次輸入想成函式參數。


### 🔗 接回第三周：工作身分與訊息 role
| 名稱 | 意義 | 本週放法 |
|---|---|---|
| 工作身分 | 助教、摘要器、分類器 | instructions 的角色描述 |
| developer | 開發者提供的高層規則 | 第三周示範；本週規則統一放 instructions |
| user | 範例輸入或本次任務 | input 列表中的訊息 |
| assistant | 提供給模型的範例答案／先前回答 | 本週用人工提供的示範答案 |
`input` 可以是字串，也可以是訊息列表。`instructions` 和訊息列表可以一起使用。
本週的 assistant 範例是事先寫好的，沒有在此新增一次 API 呼叫，也不是訓練模型。
第五周才累積真正的 user/assistant 歷史；每次只送相同固定範例不會自動記住上一輪。

In [ ]:
# 先印出請求結構；只開啟 RUN_ROLE_DEMO 才真的送出一次請求。
bridge_messages = [{"role": "user", "content": "請用一句話解釋 API。"}]
print(bridge_messages)
# 本段示範：1次API請求；不依賴後面的學生練習。
RUN_ROLE_DEMO = False
if RUN_ROLE_DEMO:
    print(ask_ai_messages(bridge_messages, "你是資管系助教，使用繁體中文與生活化例子。"))

In [ ]:
def summarize_text(text, max_chars_per_point=35, num_points=3):
    # 將可調整的輸出限制設成參數，讓同一個函式能重複用在不同資料。
    if max_chars_per_point <= 0 or num_points <= 0:
        raise ValueError("字數與重點數必須是正整數。")

    instructions = f"""
# Identity
你是精準的繁體中文摘要助理。
# Instructions
- 只根據提供的文字作答，不新增內容。
- 保留重要名稱、數字與限制。
- 輸出一個標題與 {num_points} 個條列重點。
- 每點不超過 {max_chars_per_point} 字。
"""
    # 原文放在 input；摘要規則放在 instructions，維持 prompt 結構清楚。
    user_input = f"""請摘要：
<source_text>
{text}
</source_text>"""
    return ask_ai(user_input, instructions)



# 本段示範：1次API請求；不依賴後面的學生練習。
RUN_SUMMARY_DEMO = False
if RUN_SUMMARY_DEMO:
    print(summarize_text(course_intro, max_chars_per_point=35, num_points=3))

### 🧱 Markdown 與 XML 分隔資料

- Markdown 標題讓 prompt 層次清楚、容易維護。
- XML 標籤指出資料與範例的開始、結束。
- 分隔符能降低規則與資料混淆，但不是完整安全機制；外部文字仍是不可信資料。


---

## 5️⃣ 任務拆解：把複雜要求排成步驟

當任務包含多個判斷，明確寫出可觀察的工作流程，通常比只寫「仔細思考」更容易測試。


In [ ]:
feedback = "介面很漂亮，但註冊流程一直失敗，客服兩天後才回覆。"
# Workflow 先要求模型按步驟思考，Output format 再約束最後輸出欄位。
analysis_instructions = """
# Identity
你是產品回饋分析助理。
# Workflow
1. 找出稱讚項目。2. 找出問題。3. 判斷最優先問題。4. 提出一項改善建議。
# Output format
- 優點：
- 問題：
- 優先處理：
- 建議：
"""

# 本段示範：1次API請求；不依賴後面的學生練習。
RUN_WORKFLOW_DEMO = False
if RUN_WORKFLOW_DEMO:
    print(ask_ai(f"<feedback>{feedback}</feedback>", analysis_instructions))

### 小任務：先讀規格再執行
請把任務拆解的四步對應到四個輸出欄位；自行指出一項原文無法支持的推測。
分隔符提升可讀性，不能保證防止提示注入；資料中的指令仍應視為資料。

---

## 6️⃣ Zero-shot 與 Few-shot

- **Zero-shot**：描述規則，不提供答案範例。
- **Few-shot**：提供少量且具代表性的輸入／輸出範例，讓模型掌握標籤、格式與邊界。

範例應涵蓋多樣情況；太相似的範例無法清楚呈現分類界線。


### Few-shot 的兩種呈現方式
原版以 `<example><input>…</input><output>…</output></example>` 寫進高層指令，這也是有效方式。
本版主要用 `user`／`assistant` 訊息配對，練習第三周的 list/dict 與第五周會用到的訊息結構。
兩種方式沒有保證孰優；本次實驗固定用訊息列表，只改有無範例，避免同時改兩個變因。

### 教師短示範：同一規則，三筆未見過的留言
**目標**：觀察範例是否釐清分類邊界，不要求兩版一定回答不同。
**操作**：先閱讀規則並預測三題答案；下格程式完整提供，不需修改。
開啟RUN_FEWSHOT_DEMO後執行，共3題×2版本＝6次請求。
一般題容易同樣答對；邊界題比較「增加教材說明」與「增加練習時間」，不要把多類衝突當唯一例子。
本示範不依賴尚未完成的build_few_shot_messages；學生只在下一段練習迴圈組裝。

In [ ]:
COURSE_CATEGORIES = ["內容", "進度", "設備", "其他"]
COURSE_RULES = """你是課程意見分類器，只輸出內容、進度、設備、其他其中一類。
教材內容、概念說明與例子歸內容；操作節奏與練習時間歸進度；設備或網路歸設備；其餘歸其他。
"""
# 代表性範例用明確訊息配對示範；後面的測試題沒有出現在這裡。
COURSE_EXAMPLES = [
    ("同一個概念希望用不同案例再解釋。", "內容"),
    ("觀念已懂，但還沒操作完就進下一段。", "進度"),
    ("教室網路連不上。", "設備"),
    ("希望下次自由分組。", "其他"),
]
COURSE_EXAMPLE_MESSAGES = [
    {"role": "user", "content": "同一個概念希望用不同案例再解釋。"},
    {"role": "assistant", "content": "內容"},
    {"role": "user", "content": "觀念已懂，但還沒操作完就進下一段。"},
    {"role": "assistant", "content": "進度"},
    {"role": "user", "content": "教室網路連不上。"},
    {"role": "assistant", "content": "設備"},
    {"role": "user", "content": "希望下次自由分組。"},
    {"role": "assistant", "content": "其他"},
]
COURSE_DEMO_TESTS = [
    {"input": "投影機沒有畫面。", "expected": "設備"},
    {"input": "希望多留十分鐘讓我們自己試一次。", "expected": "進度"},
    {"input": "API觀念仍不清楚，希望用完整程式再解釋一次。", "expected": "內容"},
]
RUN_FEWSHOT_DEMO = False
if RUN_FEWSHOT_DEMO:
    for case in COURSE_DEMO_TESTS:
        current_user = {"role": "user", "content": case["input"]}
        # 固定規則與測試題，只改有無示範訊息。
        zero = ask_ai_messages([current_user], COURSE_RULES).strip()
        few = ask_ai_messages(COURSE_EXAMPLE_MESSAGES + [current_user], COURSE_RULES).strip()
        print("題目：", case["input"], "｜預期：", case["expected"])
        print("zero-shot：", zero, "｜few-shot：", few)

### ✍️ 短示範觀察表
| 題型 | 預期 | zero-shot實際 | few-shot實際 | 有無改善／原因 |
|---|---|---|---|---|
| 設備一般題 | 設備 |  |  |  |
| 練習時間邊界題 | 進度 |  |  |  |
| 概念說明邊界題 | 內容 |  |  |  |
若兩版相同，記錄「本次沒有改善」，並指出範例展示了哪個分類邊界。
這三題只是示範，不足以證明few-shot的整體效果；不要刻意弱化zero-shot來製造差異。

### 引導練習：訊息組裝（必做、不付費）
**目標**：把四組(input, output)轉成8則範例訊息，再加1則本次user。
**修改位置**：只完成下一格build_few_shot_messages的三項TODO；不修改教師短示範。
**步驟**：完成函式→執行定義→下一格RUN_MESSAGE_CHECK改True→執行檢查。
**檢查**：共9則、user/assistant交錯、最後user是新問題、最後輸入不附答案。
**依賴**：此函式通過後，練習A會直接沿用，不需要另寫第二個組裝函式。

In [ ]:
def build_few_shot_messages(text, examples):
    """練習：將範例配對轉成訊息列表；此步驟不呼叫 API。"""
    require_text(text, "text")
    # TODO 1：建立空 messages 列表。
    # TODO 2：用 for 解開每個 (sample_input, sample_output)，附上 user/assistant 配對。
    # TODO 3：最後附上本次 text 的 user 訊息，回傳 messages。
    # 提示：四組範例會產生 8 則示範訊息，再加 1 則本次輸入。
    raise NotImplementedError("請完成訊息組裝練習，再開啟相關示範。")

In [ ]:
RUN_MESSAGE_CHECK = False
if RUN_MESSAGE_CHECK:
    # 此檢查不呼叫 API，完成組裝函式後可先開啟。
    messages = build_few_shot_messages("練習時間不夠。", COURSE_EXAMPLES)
    assert len(messages) == 9
    assert [m["role"] for m in messages] == ["user", "assistant"] * 4 + ["user"]
    assert messages[-1]["content"] == "練習時間不夠。"
    print(messages)

### 引導練習完成紀錄
- [ ] 9則訊息與角色順序檢查通過。
- [ ] 能指出人工範例答案與本次問題的位置。
若出現NotImplementedError，先完成函式，再進練習A；本段沒有API成本。

<div style="border-left:5px solid #f9ab00;padding:10px 14px;background:#fff8e1;">
<font color="#b06000"><b>先不要混淆</b></font>：本週只用文字規則限制輸出，仍可能出現意外格式；第 6 週再用 Structured Outputs 與 schema 做程式層級保證。
</div>


## 🛠️ 練習A：客服分類器（必做，80–110分）
**目標**：完成一個分類器，依序觀察v1無範例與v2有範例。
**先備**：引導練習build_few_shot_messages已通過離線檢查。
**規格**：四類；衝突依帳務問題→功能問題→操作疑問→其他選一類；只輸出標籤。
| 步驟 | 修改／執行位置 | 成功條件 |
|---|---|---|
| A1 | 固定資料與規則格：完成規則 | 四類定義、優先序、輸出限制 |
| A2 | 分類函式格：完成呼叫與合法標籤檢查 | 一般題只輸出一個標籤 |
| A3 | v1單筆測試格 | 先確認呼叫流程可用（1次請求） |
| A4 | 範例格：每類一例；v2單筆測試 | 未抄測試題（1次請求） |
| A5 | 教師提供的固定測試格 | 同五題比較v1/v2（10次請求） |
| A6 | 填比較表 | 說明相同、改善或退步的原因 |
v3只作選做，在完成A與擇一題後再處理。

In [ ]:
SUPPORT_LABELS = ["功能問題", "帳務問題", "操作疑問", "其他"]
# 範例與測試集分開：不要把後面的測試題直接放進範例。
SUPPORT_TESTS = [
    {"input": "我的信用卡被扣款兩次。", "expected": "帳務問題"},
    {"input": "我要在哪裡修改登入密碼？", "expected": "操作疑問"},
    {"input": "按儲存後出現錯誤，資料消失了。", "expected": "功能問題"},
    {"input": "謝謝客服快速回覆！", "expected": "其他"},
    {"input": "扣款兩次，而且報表頁面打不開。", "expected": "帳務問題"},
]

# TODO A1：寫出分類器身分、四類定義、衝突優先序與只輸出標籤的限制。
SUPPORT_RULES_V1 = ""

In [ ]:
def classify_support_message(message, examples):
            # TODO A2：用build_few_shot_messages組裝，再呼叫ask_ai_messages並strip。
            # TODO A3：輸出不在SUPPORT_LABELS時raise ValueError；合法時回傳標籤。
            raise NotImplementedError("請完成練習A分類函式。")


def support_v1(text):
    return classify_support_message(text, [])

In [ ]:
# A3：完成規則與分類函式後開啟；只測一題，1次API請求。
RUN_SUPPORT_V1_SINGLE = False
if RUN_SUPPORT_V1_SINGLE:
    print(support_v1(SUPPORT_TESTS[0]["input"]))

In [ ]:
# TODO A4：每類各寫一筆(input, label)；不要複製SUPPORT_TESTS。
SUPPORT_EXAMPLES_V2 = []


def support_v2(text):
    return classify_support_message(text, SUPPORT_EXAMPLES_V2)

# A4：完成每類一例後，先做單筆檢查；1次API請求。
RUN_SUPPORT_V2_SINGLE = False
if RUN_SUPPORT_V2_SINGLE:
    if len(SUPPORT_EXAMPLES_V2) < 4 or {label for _, label in SUPPORT_EXAMPLES_V2} != set(SUPPORT_LABELS):
        raise ValueError("請先完成每類至少一例，並使用合法標籤。")
    print(support_v2(SUPPORT_TESTS[0]["input"]))

### A5：固定測試（教師提供完整程式，不需補TODO）
傳入support_v1或support_v2是把函式交給測試器，不加括號；測試器逐題呼叫。
先完成兩版單筆測試，再把RUN_SUPPORT_COMPARISON改True，共10次API請求。
五題的預期答案已依本題規格提供，學生先核對再執行。

In [ ]:
def run_classification_tests(test_cases, classify_fn, version):
    """接受分類函式，讓同一批案例能測不同版本；保留錯誤類別便於判讀。"""
    rows = []
    for case in test_cases:
        try:
            actual = classify_fn(case["input"])
            passed = actual == case["expected"]
            status = "通過" if passed else "模型答案錯誤"
        except NotImplementedError:
            # 尚未完成練習時明確中止，不把空白答案當作分類失敗。
            raise
        except RuntimeError as exc:
            actual, passed, status = str(exc), False, "API／回應失敗"
        except ValueError as exc:
            actual, passed, status = str(exc), False, "輸入／輸出契約錯誤"
        row = dict(version=version, input=case["input"], expected=case["expected"],
                   actual=actual, passed=passed, status=status, model=MODEL)
        rows.append(row)
        print(row)
    print(f"{version}：{sum(row['passed'] for row in rows)}/{len(rows)}")
    return rows



RUN_SUPPORT_COMPARISON = False
version_results = {}
if RUN_SUPPORT_COMPARISON:
    # 必做比較只有v1/v2；先檢查範例完整，避免空範例被誤當成few-shot。
    if len(SUPPORT_EXAMPLES_V2) < 4 or {label for _, label in SUPPORT_EXAMPLES_V2} != set(SUPPORT_LABELS):
        raise ValueError("請先完成每類至少一例，再做版本比較。")
    for version, classify_fn in {"v1": support_v1, "v2": support_v2}.items():
        version_results[version] = run_classification_tests(SUPPORT_TESTS, classify_fn, version)

### A6：比較與修訂紀錄
| 版本 | 改動 | 通過 | API失敗 | 契約錯誤 | 保留／退回與原因 |
|---|---|---|---|---|---|
| v1 | 明確規則，無範例 | /5 |  |  |  |
| v2 | 相同規則＋四類範例 | /5 |  |  |  |
即使兩版同分也可完成本題；評量依據是能否說明規則、範例與測試結果。
API失敗不是prompt退步；一般合法標籤但分類錯誤，才是答案判斷問題。
- [ ] 已完成規則與合法標籤檢查。
- [ ] 已完成每類一例與v1/v2比較。
- [ ] 已記錄一個觀察或待驗證條件。

## 🔀 練習B／C擇一（110–135分）
練習A完成後，選B改寫器或C資料問答；直接到所選區塊。
每個區塊依「示範→TODO→執行→檢查→紀錄」排列；未選題跳過，課後選讀。
示範與練習分開開關，只開啟當前題，不必重跑另一題。

## ✍️ 練習B：學習內容改寫器（與C擇一）
**目標**：保留專有名詞、白話解釋、生活比喻、最後一題理解題不給答案，250字元內。
**先看示範**：下格通知改寫器完整提供；開啟RUN_REWRITE_DEMO需1次API請求，可只讀程式。
**再實作**：完成rewrite_for_beginner與模板TODO；執行定義，再開啟RUN_PRACTICE_B（1次請求）。
**檢查**：下一格字元數、專有名詞、標題；人工看事實、比喻與理解題。
**紀錄**：在B觀察表記結果；完成後直接跳到同儕覆核，不必執行C。

In [ ]:
def rewrite_notice(text, audience="大學生", tone="友善", max_chars=100):
    # audience、tone、max_chars 參數化後，同一個改寫函式可套用不同情境。
    if max_chars <= 0:
        raise ValueError("max_chars 必須是正整數。")

    instructions = f"""
# Identity
你是校園行政文字編輯。
# Instructions
- 改寫成適合{audience}閱讀的{tone}繁體中文通知。
- 不得更改日期、時間、地點與應繳項目。
- 輸出一個短標題及一段通知，總字數不超過 {max_chars} 字。
"""
    result = ask_ai(f"<original_notice>{text}</original_notice>", instructions)
    # 模型可能超出字數限制，因此仍要在程式端做第二層檢查。
    if len(result.replace("\n", "")) > max_chars:
        print(f"⚠️ 字數檢查：模型輸出超過 {max_chars} 字，請調整 prompt 或改用結構化流程。")
    return result


notice = "本課程作業二應於10月15日23:59前上傳Moodle，逾期系統關閉，不收email補交。"

RUN_REWRITE_DEMO = False
if RUN_REWRITE_DEMO:
    print(rewrite_notice(notice, audience="大學生", tone="友善", max_chars=100))

In [ ]:
# 練習 B：用 template 保留 prompt 結構，只把 max_chars 當成變數代入。
BEGINNER_REWRITE_PROMPT_TEMPLATE = """
# TODO 1：加入 Identity、Instructions、{max_chars} 參數與 Output format
"""


def rewrite_for_beginner(text, max_chars=250):
    # 提示：先檢查 max_chars，再用 template.format(max_chars=max_chars) 產生 instructions。
    # TODO 2：檢查 max_chars 必須大於 0
    # TODO 3：用 format() 代入字數，用 <technical_text> 包住資料並呼叫 ask_ai
    # TODO 4：若輸出超過 max_chars，印出警告
    # 提示：prompt 限制不是保證，程式仍需做字數檢查。
    raise NotImplementedError("請完成改寫器。")


technical_text = """
Embedding 是把文字轉換成高維度數值向量的技術。語意相近的文字，
其向量在空間中的距離通常較近，因此可以用來進行語意搜尋。
"""

RUN_PRACTICE_B = False
if RUN_PRACTICE_B:
    # TODO 5：完成函式後執行，保存結果供後面的本機檢查使用。
    raise NotImplementedError("請將本段改成呼叫並保存 beginner_result。")

In [ ]:
def check_beginner_output(result, max_chars=250):
    # 這些是可確定的格式檢查；比喻、事實與是否給出理解題答案需人工閱讀。
    return {
        "字元數符合": len(result.replace("\n", "")) <= max_chars,
        "保留專有名詞": "Embedding" in result,
        "必要標題": "## 白話說明" in result and "## 想一想" in result,
    }

if RUN_PRACTICE_B and "beginner_result" in globals():
    print(check_beginner_output(beginner_result))

### B觀察紀錄
| 原文必要事實 | 實際輸出 | 字元／標題檢查 | 人工覆核 | 修改理由 |
|---|---|---|---|---|
|  |  |  |  |  |
- [ ] 保留Embedding且解釋符合原文。
- [ ] 250字元內，有白話說明與想一想。
- [ ] 有一題理解題、不附答案；沒有新增無根據的技術細節。

## 📖 練習C：依據課程規定問答（與B擇一）
**目標**：只能根據資料回答，資料沒有答案時固定回覆資料不足。
**先看示範**：下格answer_from_context完整提供；RUN_CONTEXT_DEMO需2次API請求，可只讀程式。
**再實作**：完成answer_policy_question與問題迴圈TODO；RUN_PRACTICE_C需3次請求。
**檢查**：先核對三題預期，下一格檢查第三題固定回覆；前兩題人工對照policy。
**紀錄**：填C觀察表；完成後跳到同儕覆核，不必執行B。
policy是練習資料，不代表其他周的正式專題規定。

In [ ]:
def answer_from_context(
    question,
    context,
    missing_reply="資料不足，請詢問授課教師。",
):
    # 這是 RAG 問答的雛形：答案只能來自 context，找不到就回固定文字。
    instructions = f"""
# Identity
你是課程資訊問答助理。
# Instructions
- 只能根據 `<course_info>` 回答，不可用常識補充規定。
- 無法回答時固定回覆「{missing_reply}」
- 可以回答時先給直接答案，再附簡短依據。
"""
    # context 與 question 分開包標籤，讓模型知道哪段是資料、哪段是問題。
    data = f"<course_info>{context}</course_info><question>{question}</question>"
    return ask_ai(data, instructions)


course_rules = "作業二截止時間為10月15日23:59，繳交平台為Moodle。"

RUN_CONTEXT_DEMO = False
if RUN_CONTEXT_DEMO:
    print(answer_from_context("作業二要交到哪裡？", course_rules))
    print(answer_from_context("作業二占學期成績幾分？", course_rules))

In [ ]:
# 練習 C：policy 是唯一可用資料來源，回答不可使用常識補充。
policy = """
期中專題採個人製作，於第8週課堂展示。作品需包含簡單操作介面、
至少一次 API 呼叫與錯誤處理。展示時間每人4分鐘。
"""
questions = ["期中專題可以兩人一組嗎？", "每人展示多久？", "期中專題占總成績幾分？"]


def answer_policy_question(
    question,
    missing_reply="資料不足，請詢問授課教師。",
):
    # TODO 1：建立 instructions，只能依 policy 回答，並代入 missing_reply
    # 提示：明確要求「找不到答案時固定回覆 missing_reply」。
    # TODO 2：組合 policy 與 question，再呼叫 ask_ai
    # 提示：用 <policy> 與 <question> 分隔資料和問題。
    raise NotImplementedError("請完成資料問答。")


RUN_PRACTICE_C = False
if RUN_PRACTICE_C:
    # TODO 3：逐題呼叫並將回答依序保存到 policy_answers，供本機檢查使用。
    raise NotImplementedError("請完成問題迴圈。")

In [ ]:
if RUN_PRACTICE_C and "policy_answers" in globals():
    # 資料不足可做精確檢查；有答案的事實正確性仍需人工核對。
    assert len(policy_answers) == 3
    assert policy_answers[2].strip() == "資料不足，請詢問授課教師。"
    print("資料不足固定回覆檢查通過；請人工確認個人製作與每人4分鐘。")

### C觀察紀錄
| 問題 | 預期事實／行為 | 實際回答 | 通過／修正 |
|---|---|---|---|
| 可兩人一組嗎？ | 個人製作 |  |  |
| 展示多久？ | 每人4分鐘 |  |  |
| 占總成績幾分？ | 固定資料不足 |  |  |
- [ ] 有答案時忠於policy，措辭可不同。
- [ ] 沒答案時不猜測，固定回覆通過程式檢查。

### 同儕檢查（135–150分）
交換規則與測試紀錄：對方能否不用猜測就判斷衝突標籤？範例是否誤把測試答案洩漏？
指出一個失敗或仍未驗證的條件，完成一次修改並寫下原因。

## 🌱 延伸選做：v3增加衝突範例
完成練習A及B/C其中一題後再做；不是課堂完成門檻。
保留v2的規則與範例，只增加一筆符合優先序的衝突範例。
不抄測試題；RUN_SUPPORT_V3=True以相同五題再測，需5次API請求。

In [ ]:
# TODO 選做：保留v2範例並加入一筆衝突範例。
SUPPORT_EXAMPLES_V3 = []


def support_v3(text):
    return classify_support_message(text, SUPPORT_EXAMPLES_V3)

RUN_SUPPORT_V3 = False
if RUN_SUPPORT_V3:
    if SUPPORT_EXAMPLES_V3[:len(SUPPORT_EXAMPLES_V2)] != SUPPORT_EXAMPLES_V2 or len(SUPPORT_EXAMPLES_V3) <= len(SUPPORT_EXAMPLES_V2):
        raise ValueError("請保留v2範例並至少增加一筆衝突範例。")
    version_results["v3"] = run_classification_tests(SUPPORT_TESTS, support_v3, "v3")

| 選做版本 | 修改 | 通過 | 保留／退回與原因 |
|---|---|---|---|
| v3 | 在v2增加衝突範例 | /5 |  |
v3也可能與v2同分；記錄觀察，不宣稱必然改善。

## ✅ 課堂完成檢核
- [ ] 理解工作身分與API role、五元素、資料分隔與任務拆解。
- [ ] 引導練習：9則訊息與最後user檢查通過。
- [ ] 練習A：客服分類器合法標籤、每類一例、v1/v2五題比較與紀錄。
- [ ] 練習B或C：完成所選函式、程式檢查與人工覆核。
- [ ] 能說明相同結果也可能合理，並指出範例提供了什麼資訊。
v3、未選題與額外挑戰都是選做，不列入課堂必做檢核。

## 📝 課後繳交與第五周銜接
必交：訊息組裝結果、客服v1/v2規則／範例、五題比較表、B或C其中一題的輸出與檢查，
模型名稱及150字修訂反思。v3與未選題可另附選做成果，不要求全部完成。
不提交金鑰、個資或其他敏感資料。
第五周把人工user/assistant範例延伸成真實對話history；第六周再教Structured Outputs。

## 📚 延伸閱讀

- OpenAI Prompt engineering：https://developers.openai.com/api/docs/guides/prompt-engineering
- OpenAI Text generation：https://developers.openai.com/api/docs/guides/text

下週將進入：**多輪對話、conversation state 與 Streaming 輸出**。
